# 01 LSLR Clean

This notebook cleans the LSLR replacement file.

Important rule:
- Do not infer or fill missing values.
- Only standardize columns, convert types, and prepare clean outputs.

The input file must be in the same folder as this notebook.

In [13]:
from pathlib import Path

import pandas as pd
from IPython.display import display

BASE_DIR = Path('.').resolve()
OUTPUT_DIR = BASE_DIR / 'LSLR_clean_output'
OUTPUT_DIR.mkdir(exist_ok=True)

lslr_file_name = input(
    'Enter the LSLR workbook file name in this folder: '
).strip()

if not lslr_file_name:
    raise ValueError('You must enter an LSLR workbook file name.')

LSLR_FILE_PATH = BASE_DIR / lslr_file_name

if not LSLR_FILE_PATH.exists():
    raise FileNotFoundError(f'File not found: {LSLR_FILE_PATH}')

print('LSLR file:', LSLR_FILE_PATH)
print('Output folder:', OUTPUT_DIR)

LSLR file: D:\Study Experience\UMich\Umich 2026 Winter\SI 699\Data Cleaning\LSLR_Inventory_Pipeline\2024-2025-LSLR-Data.xlsx
Output folder: D:\Study Experience\UMich\Umich 2026 Winter\SI 699\Data Cleaning\LSLR_Inventory_Pipeline\LSLR_clean_output


In [14]:
LSLR_COLUMNS = [
    'pwsid',
    'supply_name',
    'county',
    'year',
    'lines_replaced',
]


def load_lslr_workbook(path):
    df = pd.read_excel(path, header=2)
    df = df.iloc[:, 0:5].copy()
    df.columns = LSLR_COLUMNS

    df['pwsid'] = df['pwsid'].astype(str).str.strip()
    df['supply_name'] = df['supply_name'].astype(str).str.strip()
    df['county'] = df['county'].astype(str).str.strip()
    df['year'] = pd.to_numeric(df['year'], errors='coerce')
    df['lines_replaced'] = pd.to_numeric(df['lines_replaced'], errors='coerce')

    raw_rows = len(df)

    df = df.dropna(subset=['pwsid', 'year']).copy()
    df['year'] = df['year'].astype('Int64')

    exact_duplicates = df[df.duplicated(keep=False)].sort_values(
        ['pwsid', 'year', 'county', 'supply_name']
    ).reset_index(drop=True)

    key_duplicates = df[
        df.duplicated(subset=['pwsid', 'year'], keep=False)
    ].sort_values(['pwsid', 'year', 'county', 'supply_name']).reset_index(drop=True)

    clean_raw = df.drop_duplicates().reset_index(drop=True)

    clean_grouped = (
        clean_raw.groupby(['pwsid', 'year', 'county'], as_index=False)['lines_replaced']
        .sum(min_count=1)
        .rename(columns={'county': 'county_for_checking'})
        .sort_values(['pwsid', 'year', 'county_for_checking'])
        .reset_index(drop=True)
    )

    summary = pd.DataFrame([
        {
            'raw_rows': raw_rows,
            'rows_after_drop_missing_keys': len(df),
            'exact_duplicate_rows': len(exact_duplicates),
            'pwsid_year_duplicate_rows': len(key_duplicates),
            'clean_raw_rows': len(clean_raw),
            'clean_grouped_rows': len(clean_grouped),
        }
    ])

    return clean_raw, clean_grouped, exact_duplicates, key_duplicates, summary

In [15]:
clean_raw, clean_grouped, exact_duplicates, key_duplicates, summary_df = load_lslr_workbook(LSLR_FILE_PATH)

display(summary_df)
display(clean_raw.head())
display(key_duplicates.head())

,raw_rows,rows_after_drop_missing_keys,exact_duplicate_rows,pwsid_year_duplicate_rows,clean_raw_rows,clean_grouped_rows
0,916,916,0,4,916,914


,pwsid,supply_name,county,year,lines_replaced
0,MI0000040,ADRIAN,LENAWEE,2021,57
1,MI0000120,ALLEGAN,ALLEGAN,2021,12
2,MI0000130,ALLEN PARK,WAYNE,2021,8
3,MI0000150,"ALMONT, VILLAGE OF",LAPEER,2021,7
4,MI0000160,"ALPENA, CITY OF",ALPENA,2021,25


,pwsid,supply_name,county,year,lines_replaced
0,MI0003525,KALAMAZOO LAKE SEWER & WATER AUTHORITY - City ...,ALLEGAN,2022,6
1,MI0003525,KALAMAZOO LAKE SEWER & WATER AUTHORITY- City o...,ALLEGAN,2022,7
2,MI0003525,KALAMAZOO LAKE SEWER & WATER AUTHORITY - City ...,ALLEGAN,2023,17
3,MI0003525,KALAMAZOO LAKE SEWER & WATER AUTHORITY - KLSWA,ALLEGAN,2023,1


In [16]:
output_stem = Path(lslr_file_name).stem

raw_path = OUTPUT_DIR / f'{output_stem}_raw_clean.csv'
grouped_path = OUTPUT_DIR / f'{output_stem}_grouped_clean.csv'
exact_dup_path = OUTPUT_DIR / f'{output_stem}_exact_duplicates.csv'
key_dup_path = OUTPUT_DIR / f'{output_stem}_pwsid_year_duplicates.csv'

clean_raw.to_csv(raw_path, index=False)
clean_grouped.to_csv(grouped_path, index=False)
exact_duplicates.to_csv(exact_dup_path, index=False)
key_duplicates.to_csv(key_dup_path, index=False)

print('Saved:', raw_path)
print('Saved:', grouped_path)
print('Saved:', exact_dup_path)
print('Saved:', key_dup_path)

Saved: D:\Study Experience\UMich\Umich 2026 Winter\SI 699\Data Cleaning\LSLR_Inventory_Pipeline\LSLR_clean_output\2024-2025-LSLR-Data_raw_clean.csv
Saved: D:\Study Experience\UMich\Umich 2026 Winter\SI 699\Data Cleaning\LSLR_Inventory_Pipeline\LSLR_clean_output\2024-2025-LSLR-Data_grouped_clean.csv
Saved: D:\Study Experience\UMich\Umich 2026 Winter\SI 699\Data Cleaning\LSLR_Inventory_Pipeline\LSLR_clean_output\2024-2025-LSLR-Data_exact_duplicates.csv
Saved: D:\Study Experience\UMich\Umich 2026 Winter\SI 699\Data Cleaning\LSLR_Inventory_Pipeline\LSLR_clean_output\2024-2025-LSLR-Data_pwsid_year_duplicates.csv
